# [M2] DẠY MÁY HỌC TỪ DỮ LIỆU
## Buổi S03: Phân Loại Nhị Phân & Cây Quyết Định (Logistic Regression & Decision Trees)

---

### Metadata Khóa Học
* **Module:** M2 — DẠY MÁY HỌC TỪ DỮ LIỆU
* **Buổi:** S03
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook huấn luyện Decision Tree và trích xuất Feature Importance
* **Mức độ liên quan VOAI / Olympic AI:** Mô hình nền tảng cho dữ liệu bảng thi đấu, khám phá các đặc trưng quan trọng nhất.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu cách chuyển hóa từ bài toán Hồi quy sang Phân loại nhị phân thông qua hàm kích hoạt Sigmoid và mô hình Logistic Regression.
- Nắm vững cơ chế hoạt động của Cây Quyết Định (Decision Tree): Chia nhánh dựa trên độ vẩn đục Gini (Gini Impurity) hoặc Entropy.
- Quan sát trực quan ranh giới quyết định (Decision Boundary) và hiện tượng quá khớp (Overfitting) khi cây phát triển quá sâu.
- Hiểu trực giác mở rộng sang Rừng Ngẫu Nhiên (Random Forest Ensemble) và trích xuất độ quan trọng đặc trưng (Feature Importance).

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M2_S02 (Hồi quy tuyến tính & Loss function).
- Hiểu khái niệm xác suất nằm trong khoảng [0, 1].

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong thực tế, đa số các bài toán kinh doanh và thi đấu AI là bài toán **Phân loại (Classification)**: Dự đoán khách hàng có rời bỏ dịch vụ không, khối u là lành tính hay ác tính, người dùng có nhấp vào quảng cáo không. Đường thẳng của Hồi quy tuyến tính có thể cho ra kết quả âm hoặc lớn hơn 1, hoàn toàn không phù hợp để biểu diễn xác suất!

Logistic Regression và Decision Tree là hai trụ cột không thể thay thế: Một bên đại diện cho ranh giới phân tách tuyến tính êm dịu dựa trên xác suất, một bên đại diện cho cấu trúc suy luận dạng cây trực quan, có khả năng xử lý phi tuyến tính mà không cần giả định phân phối dữ liệu.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Logistic Regression & Hàm Sigmoid
* Để đưa giá trị đầu ra $z = \mathbf{w}^T \mathbf{x} + b$ về khoảng xác suất $[0, 1]$, ta sử dụng hàm Sigmoid:
  $$\sigma(z) = \frac{1}{1 + e^{-z}}$$
* **Ngưỡng quyết định (Decision Threshold):** Thông thường, nếu $\sigma(z) \ge 0.5 \implies \hat{y} = 1$, ngược lại $\hat{y} = 0$.
* **Hàm mất mát (Binary Cross-Entropy / Log-Loss):** Phạt cực nặng nếu mô hình tự tin dự đoán xác suất gần 1 cho một mẫu thực tế có nhãn 0!

### 2. Cây Quyết Định (Decision Tree) & Chỉ số Gini Impurity
* Cây phân nhánh dữ liệu bằng các câu hỏi nhị phân dạng `Feature <= Threshold`.
* Tại mỗi nút (Node), thuật toán duyệt qua mọi đặc trưng và mọi ngưỡng chia để tìm ra nhát cắt làm giảm **Độ vẩn đục (Gini Impurity)** nhiều nhất:
  $$Gini(p) = 1 - \sum_{k=1}^K p_k^2$$
  (Gini = 0 khi nút hoàn toàn thuần khiết - tất cả các mẫu đều cùng một lớp).
* **Hiểm họa Overfitting:** Nếu để cây phát triển tự do không giới hạn độ sâu (`max_depth=None`), cây sẽ chia nhánh cho đến khi mỗi chiếc lá chỉ chứa đúng 1 mẫu dữ liệu -> Học vẹt 100% tập Train nhưng thất bại trên tập Test!



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Cấu trúc Cây Quyết Định (Decision Tree):
              [ Glucose <= 125.5 ]
                   /        \
                 Yes         No
                 /            \
        [ BMI <= 28.3 ]      [ Nút lá: 95% Mắc bệnh ]
           /       \
         Yes        No
         /           \
[ Lá: 5% Bệnh ]   [ Lá: 60% Bệnh ]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# 1. Tao tap du lieu phan loai 2 dac trung de de dang truc quan hoa mat phang
X, y = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, flip_y=0.05, class_sep=1.2, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# 2. Huan luyen Logistic Regression
clf_lr = LogisticRegression()
clf_lr.fit(X_train, y_train)

# 3. Huan luyen Decision Tree (Co gioi han do sau de tranh Overfitting)
clf_dt = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_dt.fit(X_train, y_train)

# 4. Huan luyen Random Forest
clf_rf = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
clf_rf.fit(X_train, y_train)

print(f"Logistic Regression Train Acc: {clf_lr.score(X_train, y_train):.3f} | Test Acc: {clf_lr.score(X_test, y_test):.3f}")
print(f"Decision Tree (d=3) Train Acc: {clf_dt.score(X_train, y_train):.3f} | Test Acc: {clf_dt.score(X_test, y_test):.3f}")
print(f"Random Forest (d=3) Train Acc: {clf_rf.score(X_train, y_train):.3f} | Test Acc: {clf_rf.score(X_test, y_test):.3f}")

### 🔍 Code Walkthrough (Khối 1)
Khối code tạo lập bài toán phân loại 2D có nhiễu thực tế:
- Huấn luyện đồng thời 3 mô hình cốt lõi: Logistic Regression, Decision Tree (khống chế `max_depth=3`) và Random Forest.
- Đo lường và so sánh điểm số chính xác (Accuracy) trên cả tập Train và tập Test.


In [ ]:
# 5. Truc quan hoa cau truc cay quyet dinh va Do quan trong dac trung
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# So do cay quyet dinh
plot_tree(clf_dt, feature_names=['Feature_1', 'Feature_2'], class_names=['Class_0', 'Class_1'], 
          filled=True, rounded=True, ax=axes[0], fontsize=9)
axes[0].set_title('Cau truc suy luan cua Decision Tree (max_depth=3)', fontweight='bold')

# Do quan trong cua dac trung (Feature Importances) tu Random Forest
importances = clf_rf.feature_importances_
axes[1].bar(['Feature_1', 'Feature_2'], importances, color=['#2b5c8f', '#d62728'], width=0.4)
axes[1].set_title('Do quan trong dac trung (Random Forest Feature Importances)', fontweight='bold')
axes[1].set_ylabel('Importance Score')
axes[1].set_ylim(0, 1.0)

plt.tight_layout()
plt.show()

### 🔍 Code Walkthrough (Khối 2)
Khối code minh họa tính khả giải thích cao (Interpretability) của mô hình cây:
- `plot_tree`: Cho phép quan sát từng câu hỏi điều kiện tại mỗi nút nhánh và độ tinh khiết Gini.
- `feature_importances_`: Cho biết đặc trưng nào đóng góp nhiều nhất vào việc giảm độ vẩn đục toàn hệ thống.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Thử nghiệm Overfitting với `max_depth`
Huấn luyện một cây quyết định mới với `max_depth=None` (không giới hạn) và `min_samples_split=2`. Quan sát điểm số trên tập Train và tập Test. Nêu nhận xét về hiện tượng Overfitting.

### Bài 2 (Easy): Đọc xác suất dự đoán `predict_proba`
Sử dụng phương thức `clf_lr.predict_proba(X_test[:5])` để in ra xác suất dự đoán chi tiết cho 5 mẫu đầu tiên. Kiểm tra xem tổng xác suất của 2 cột có bằng 1.0 hay không.

### Bài 3 (Medium): Thay đổi ngưỡng quyết định (Threshold Tuning)
Mặc định ngưỡng phân loại là 0.5. Viết một đoạn code gán nhãn 1 nếu xác suất $P(y=1) \ge 0.3$. Thao tác này có lợi ích gì khi ta muốn bắt triệt để các ca bệnh nguy hiểm (ưu tiên Recall)?

### Bài 4 (Medium): So sánh tiêu chí chia nhánh Gini vs Entropy
Huấn luyện hai cây quyết định có cùng độ sâu `max_depth=4`, một cây dùng `criterion='gini'` và một cây dùng `criterion='entropy'`. So sánh độ chính xác và cấu trúc của hai cây.

### Bài 5 (Challenge): Vẽ đường ranh giới quyết định (Decision Boundary) 2D
Sử dụng `np.meshgrid` và `plt.contourf` để vẽ ranh giới quyết định 2D cho cả Logistic Regression (đường thẳng tuyến tính) và Decision Tree (các đường gấp khúc vuông góc theo trục).



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong các bảng xếp hạng Olympic AI dạng dữ liệu bảng (Tabular Data), các mô hình họ cây (Decision Tree, Random Forest, GBDT) luôn là "ông hoàng" thống trị. Chúng không đòi hỏi chuẩn hóa đặc trưng (Feature Scaling), có khả năng tự động xử lý tương quan phi tuyến, và cung cấp ngay lập tức bảng **Feature Importance** giúp thí sinh biết đặc trưng nào đang mang lại giá trị cao nhất để tiếp tục đào sâu kỹ nghệ đặc trưng.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Một bạn sinh viên dùng Decision Tree huấn luyện trên tập dữ liệu bị mất cân bằng trầm trọng (95% nhãn 0, 5% nhãn 1) nhưng không đặt tham số `class_weight`, khiến cây luôn dự đoán ra nhãn 0 để đạt Gini thấp:

```python
# CODE CÓ VẤN ĐỀ TRỌNG SỐ LỚP:
# Khi du lieu mat can bang nang, cay quyet dinh se bo qua hoan toan lop thieu so!
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao việc tối ưu Gini Impurity thuần túy có thể bỏ rơi lớp thiểu số.
2. Sửa lỗi bằng cách thêm tham số `class_weight='balanced'` trong Scikit-Learn để tự động phạt nặng các lỗi sai trên lớp hiếm.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Hàm Sigmoid biến đổi bất kỳ giá trị số thực nào $z \in (-\infty, +\infty)$ về khoảng giá trị nào?
   - A. $[-1, 1]$
   - B. $[0, 1]$
   - C. $[0, +\infty)$
   - D. $[-\pi, \pi]$

2. Độ tinh khiết Gini (Gini Impurity) của một nút lá bằng 0 khi nào?
   - A. Khi nút đó chứa số lượng mẫu bằng nhau giữa các lớp.
   - B. Khi tất cả các mẫu trong nút lá đó đều thuộc về cùng một lớp duy nhất (Pure node).
   - C. Khi nút đó không chứa mẫu nào.
   - D. Khi cây đạt độ sâu tối đa.

3. Kỹ thuật nào sau đây giúp kiểm soát trực tiếp hiện tượng Overfitting trong Cây Quyết Định?
   - A. Tăng số lượng mẫu lên vô hạn.
   - B. Cắt tỉa cây (Pruning) hoặc giới hạn độ sâu tối đa bằng siêu tham số `max_depth`.
   - C. Bỏ qua bước chia Train/Test.
   - D. Xóa nhãn của dữ liệu.

4. Tại sao ranh giới quyết định (Decision Boundary) của một Cây Quyết Định luôn có dạng các đường thẳng song song với các trục tọa độ?
   - A. Vì cây chỉ sử dụng hàm sin và cosin.
   - B. Vì tại mỗi bước phân nhánh, cây chỉ so sánh một đặc trưng đơn lẻ với một giá trị ngưỡng (`Feature_i <= Threshold`).
   - C. Do lỗi hiển thị của Matplotlib.
   - D. Vì dữ liệu đã được chuẩn hóa Min-Max.

5. Rừng Ngẫu Nhiên (Random Forest) cải thiện độ ổn định và giảm phương sai (Variance) của một cây quyết định đơn lẻ dựa trên nguyên lý cốt lõi nào?
   - A. Huấn luyện duy nhất một cây với kích thước khổng lồ.
   - B. Tạo ra nhiều cây quyết định độc lập trên các tập con dữ liệu ngẫu nhiên (Bootstrap Aggregating / Bagging) và lấy biểu quyết số đông (Ensemble Voting).
   - C. Bỏ qua hoàn toàn dữ liệu có nhiễu.
   - D. Luôn luôn chọn đặc trưng đầu tiên.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Bản chất hàm Sigmoid và mô hình Logistic Regression trong phân loại nhị phân.
  - Cơ chế phân nhánh theo độ vẩn đục Gini Impurity của Cây Quyết Định.
  - Nhận diện trực quan hiện tượng Overfitting qua độ sâu của cây (`max_depth`).
  - Trực giác về Random Forest và trích xuất độ quan trọng đặc trưng (Feature Importance).
* **What I can now do:**
  - Huấn luyện và trực quan hóa cây suy luận quyết định trên dữ liệu thực tế.
  - Điều chỉnh `max_depth` và trích xuất đặc trưng then chốt của bài toán.
* **What I still need to practice:**
  - Học cách đo lường độ chính xác đa chiều: **Confusion Matrix, Precision, Recall, F1-Score, ROC-AUC** ở Buổi 4.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M2_S04**, chúng ta sẽ đi sâu vào "trọng tài" của mọi cuộc thi: **Đọc & Hiểu Toàn Diện Metric Đánh Giá Mô Hình** — Bóc trần nghịch lý Accuracy, phân tích ma trận nhầm lẫn (Confusion Matrix), và làm chủ cán cân Precision-Recall!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M2**:
- Logistic regression & Sigmoid activation ✓
- Decision trees & Gini impurity ✓
- Overfitting control via max_depth & Random Forest intuition ✓

* **Session Deliverable:** Mã nguồn huấn luyện phân loại nhị phân và trích xuất Feature Importance
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
